# 07a · What changes when we review the answer key?

The reviewer assessed 20 development examples. We use the **19 accepted decisions** and exclude the one marked unclear. This notebook recalculates all eight candidates with **the same saved predictions** against historical tags and the reviewed answers. It makes no provider calls and does not train a model.

The reference is `dr43-reviewed-v1`, associated with rubric-v2. The reviewer’s accepted interpretations are retained, including documented source and rule limitations. This targeted review is not an independent final test or an estimate of production label quality.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))
from demo_logic.evaluation import multilabel_metrics
from notebooks.tools import file_sha256, percent, show_table, verify_sha256

folder = root / "reports/development-review/2026-10-01"
reference_path = folder / "approved-reference-v1.json"
REFERENCE_SHA256 = "29c184df9da29178b033d6e3ba37c22f2b6a34fef2b49ed038bd2255c688f4a7"
verify_sha256(reference_path, REFERENCE_SHA256)
reference = json.loads(reference_path.read_text())
verify_sha256(folder / "manifest.json", reference["manifest_sha256"])
manifest = json.loads((folder / "manifest.json").read_text())
for path, expected in manifest["sources"].items():
    verify_sha256(root / path, expected)
records = {r["external_id"]: r for r in manifest["records"]}
reviewed = reference["records"]
ids = [r["external_id"] for r in reviewed]
assert len(ids) == len(set(ids)) == 19
assert set(ids).isdisjoint(r["external_id"] for r in reference["excluded"])
assert set(ids).issubset(manifest["ordered_ids"])
show_table(
    ["Reference fact", "Value"],
    [
        ("Version", reference["reference_version"]),
        ("Accepted examples", len(reviewed)),
        ("Unclear examples excluded", len(reference["excluded"])),
        ("Final-test examples used", reference["final_test_records_used"]),
    ],
)

## 1. Recalculate the scores

Only the reference tags change. Predictions, event IDs and tag-selection policies stay fixed. Three-tag human answers stay complete; we do not trim them to favor a two-tag model. Coverage therefore stays unchanged. Precision, recall and F1 can change because the answers used for scoring changed.


In [ ]:
results = {}
for key, name in manifest["models"].items():
    predictions = [records[r["external_id"]]["predictions"][key] for r in reviewed]
    old = multilabel_metrics(
        [records[r["external_id"]]["historical"] for r in reviewed],
        predictions,
        reference["tag_slugs"],
    )
    new = multilabel_metrics([r["tags"] for r in reviewed], predictions, reference["tag_slugs"])
    results[key] = {"model": name, "historical": old, "reviewed": new}
show_table(
    [
        "Model",
        "Precision: old → reviewed",
        "Recall: old → reviewed",
        "F1: old → reviewed",
        "Coverage",
    ],
    [
        (
            v["model"],
            *(
                f"{percent(v['historical'][metric])} → {percent(v['reviewed'][metric])}"
                for metric in ["micro_precision", "micro_recall", "micro_f1"]
            ),
            percent(v["reviewed"]["coverage"]),
        )
        for v in results.values()
    ],
)
saved = json.loads((folder / "approved-comparison-v1.json").read_text())
assert saved["reference_sha256"] == REFERENCE_SHA256
for key, values in results.items():
    assert values["historical"] == saved["models"][key]["historical"]
    assert values["reviewed"] == saved["models"][key]["reviewed"]
print("Recalculated metrics match the saved comparison.")

In [ ]:
labels = [v["model"] for v in results.values()]
y = list(range(len(labels)))
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(
    [i + 0.18 for i in y],
    [v["historical"]["micro_f1"] * 100 for v in results.values()],
    height=0.36,
    label="Historical answers",
    color="#8899aa",
)
ax.barh(
    [i - 0.18 for i in y],
    [v["reviewed"]["micro_f1"] * 100 for v in results.values()],
    height=0.36,
    label="Reviewed reference",
    color="#2a8b84",
)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Micro F1 (%)")
ax.set_title("Same 19 development events and predictions; different answer key")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## 2. What did the pivot reveal?

- **Historical answers:** 12 of the 19 accepted tag sets changed. Because we deliberately selected disagreements, this does not mean 12/19 production labels are wrong.
- **Rules:** Pride now supports `lgbtq` + `human-rights`; a protest focused on a German bill can also support `politics`. These are explicit rubric-v2 decisions, not hidden exceptions.
- **Missing information:** the reviewer needed background for six of the 20 cases. This is a reason to test evidence enrichment separately; it does not prove which model errors research would fix.
- **Model disagreements:** The models still omit accepted tags or suggest different ones. Inspect the table below. These are measured differences from the approved reference, not independently proven mistakes in every case.

GPT-OSS precision rises from 66.7% to 75.0%, and Qwen from 53.6% to 71.4%. Their recall remains below 80%. Regex and the original NLI lose F1 under the reviewed answers. Reviewing labels helps us understand the scores; it does not improve every method.


In [ ]:
show_table(
    ["Model", "Extra suggested tags", "Missed reference tags", "Exact matches / 19"],
    [
        (
            v["model"],
            sum(t["fp"] for t in v["reviewed"]["per_tag"].values()),
            sum(t["fn"] for t in v["reviewed"]["per_tag"].values()),
            round(v["reviewed"]["exact_match"] * len(reviewed)),
        )
        for v in results.values()
    ],
)
show_table(
    ["Case", "Retained limitation"],
    [(r["case"], r["limitation"]) for r in reviewed if r["limitation"]],
)

## 3. Decision and limits

the reviewer’s accepted choices are the reference for this diagnostic. Cases with evidence or interpretation warnings remain included; the warnings are visible above. Case 19 remains excluded as unclear. We do not describe this as a fully independent or uniformly verified rubric-v2 benchmark.

**Decision:** Keep the before/after results and resume the planned candidate comparison. A small second review would help measure agreement, but is not a prerequisite for continuing. Before final evaluation, review only held-out labels affected by the changed rules without viewing final predictions. Do not use final answers to tune models, prompts or thresholds.

Source review, reference and row-level results stay in Git-ignored `reports/`. Commit this notebook with empty outputs. This run uses no new inference and no API spend.


## 4. Ten random follow-up cases — DR-44

We selected ten cases randomly from the remaining 74 development-check events, excluding all first-batch cases, before inspecting disagreements. the reviewer accepted nine and marked one unclear. We score the **same nine** against historical and reviewed answers, then show a separate **28-event pooled comparison** (19 + 9). Two unclear cases stay excluded. This is random within the remaining development pool; the pooled sample includes a targeted first batch and is not uniformly random.

Accepted tags are retained. A source attached to the homelessness case was unrelated to the event; the title supports the topic, and the source warning is recorded. The unclear commemoration case stays excluded rather than being guessed. No new inference or final-test answers are used.


In [ ]:
followup_folder = root / "reports/development-review/2026-10-01-random10"
followup_reference_path = followup_folder / "random10-approved-reference-v1.json"
verify_sha256(
    followup_reference_path, "fb7691b0276907793449cbb0a6702e7c7bc7d645775b363e1913cf74f653b83e"
)
followup_reference = json.loads(followup_reference_path.read_text())
verify_sha256(followup_folder / "random10-manifest.json", followup_reference["manifest_sha256"])
followup_manifest = json.loads((followup_folder / "random10-manifest.json").read_text())
followup_saved = json.loads((followup_folder / "random10-approved-comparison-v1.json").read_text())
assert followup_saved["reference_sha256"] == file_sha256(followup_reference_path)
assert followup_saved["first_reference_sha256"] == REFERENCE_SHA256
assert set(followup_manifest["ordered_ids"]).isdisjoint(manifest["ordered_ids"])
for path, expected in followup_manifest["sources"].items():
    verify_sha256(root / path, expected)
all_records = records | {r["external_id"]: r for r in followup_manifest["records"]}
followup_results = {}
for batch, accepted in [
    ("random10", followup_reference["records"]),
    ("combined", reviewed + followup_reference["records"]),
]:
    assert len({r["external_id"] for r in accepted}) == len(accepted)
    batch_results = {}
    for key, name in manifest["models"].items():
        guesses = [all_records[r["external_id"]]["predictions"][key] for r in accepted]
        old = multilabel_metrics(
            [all_records[r["external_id"]]["historical"] for r in accepted],
            guesses,
            reference["tag_slugs"],
        )
        new = multilabel_metrics([r["tags"] for r in accepted], guesses, reference["tag_slugs"])
        assert old == followup_saved["batches"][batch]["models"][key]["historical"]
        assert new == followup_saved["batches"][batch]["models"][key]["reviewed"]
        batch_results[key] = {"model": name, "historical": old, "reviewed": new}
    followup_results[batch] = batch_results
    print(f"{batch}: {len(accepted)} scored events")
    show_table(
        [
            "Model",
            "Precision: old → reviewed",
            "Recall: old → reviewed",
            "F1: old → reviewed",
            "Coverage",
        ],
        [
            (
                v["model"],
                *(
                    f"{percent(v['historical'][metric])} → {percent(v['reviewed'][metric])}"
                    for metric in ["micro_precision", "micro_recall", "micro_f1"]
                ),
                percent(v["reviewed"]["coverage"]),
            )
            for v in batch_results.values()
        ],
    )
print("Both batches reproduce the saved metrics; earlier references remain unchanged.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True)
for ax, (batch, values) in zip(axes, followup_results.items()):
    names = [v["model"] for v in values.values()]
    positions = list(range(len(names)))
    ax.barh(
        [i + 0.18 for i in positions],
        [v["historical"]["micro_f1"] * 100 for v in values.values()],
        height=0.36,
        label="Historical answers",
        color="#8899aa",
    )
    ax.barh(
        [i - 0.18 for i in positions],
        [v["reviewed"]["micro_f1"] * 100 for v in values.values()],
        height=0.36,
        label="Reviewed answers",
        color="#2a8b84",
    )
    ax.set_yticks(positions, names if batch == "random10" else [])
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Micro F1 (%)")
    ax.set_title("Random follow-up: 9 accepted" if batch == "random10" else "Combined: 28 accepted")
axes[0].legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Follow-up finding and next step

**The extra cases did not improve every score.** Three of nine accepted tag sets changed. Five of ten cases needed background, based on the reviewer’s review. One unclear case remains excluded; it does not require another tag just because the task is multi-label.

On the random batch, Qwen precision changes from **69.2% to 84.6%** and recall from **81.8% to 91.7%**. GPT-OSS precision stays **81.8%**, while recall changes from **81.8% to 75.0%**. This is only nine scored events; it is not evidence that Qwen meets deployment targets generally.

Across all 28 accepted events, reviewed GPT-OSS achieves **77.1% precision, 61.4% recall and 68.4% F1**; Qwen achieves **75.6%, 70.5% and 72.9%**. Neither meets both 80% targets. All eight candidate rows remain available above, including methods whose scores deteriorate.

**Decision:** The bounded pivot check is complete. Keep the reference versions and limitations, then resume Jev. Do not expand the review indefinitely to seek a better score. Final evaluation and any affected-label review remain separate from tuning. No new model calls or API spend were needed.
